# 🖥️ Laptop Price Prediction
## Step 6: Model Training
**Student:** Shivakumar | B.Tech CSE | VEMU Institute of Technology

---
## 📦 0. Install & Import Libraries

In [ ]:
!pip install xgboost scikit-learn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12})
os.makedirs('models', exist_ok=True)
print("All libraries imported")

---
## 📂 1. Load Data & Rebuild Preprocessing

In [ ]:
df = pd.read_csv('laptop_price_featured.csv')
print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} cols")
df.head(3)

In [ ]:
TARGET    = 'log_price'
DROP_COLS = ['Price_euros','log_price','Ram','Weight','Cpu','Gpu','Memory','ScreenResolution','OpSys']
drop_existing = [c for c in DROP_COLS if c in df.columns]
X = df.drop(columns=drop_existing)
y = df[TARGET]
y_actual = df['Price_euros']

numerical_cols   = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()
binary_cols      = [c for c in numerical_cols if X[c].nunique() == 2]
numerical_cols   = [c for c in numerical_cols if c not in binary_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
_, _, y_train_actual, y_test_actual = train_test_split(X, y_actual, test_size=0.2, random_state=42)

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline,   numerical_cols),
    ('cat', categorical_pipeline, categorical_cols),
    ('bin', 'passthrough',        binary_cols),
], remainder='drop')

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

selector = SelectKBest(f_regression, k=20)
X_train_sel = selector.fit_transform(X_train_proc, y_train)
X_test_sel  = selector.transform(X_test_proc)

print(f"Train: {X_train_sel.shape}  Test: {X_test_sel.shape}")
print("Preprocessing complete")

---
## 📋 2. Model Selection Justification

In [ ]:
print('''
Model 1: Ridge Regression (Baseline)
  - Linear model with L2 regularization
  - Fast, interpretable baseline
  - Handles multicollinearity from OHE
  - Expected R2: ~0.70-0.75

Model 2: Random Forest Regressor
  - Ensemble of decision trees (bagging)
  - Handles non-linear relationships
  - Robust to outliers
  - Expected R2: ~0.85-0.88

Model 3: XGBoost Regressor
  - Gradient boosting — learns from errors
  - Best on tabular structured data
  - Built-in L1+L2 regularization
  - Expected R2: ~0.88-0.92
''')

---
## 🔵 Model 1: Ridge Regression

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train_sel, y_train)
cv_scores_ridge = cross_val_score(ridge, X_train_sel, y_train, cv=cv, scoring='r2')

y_pred_ridge_log  = ridge.predict(X_test_sel)
y_pred_ridge_euro = np.expm1(y_pred_ridge_log)

print(f"Ridge CV R2: {cv_scores_ridge.mean():.4f} +/- {cv_scores_ridge.std():.4f}")
print("Ridge trained")

---
## 🟢 Model 2: Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=None,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
rf.fit(X_train_sel, y_train)
cv_scores_rf = cross_val_score(rf, X_train_sel, y_train, cv=cv, scoring='r2')

y_pred_rf_log  = rf.predict(X_test_sel)
y_pred_rf_euro = np.expm1(y_pred_rf_log)

print(f"RF CV R2: {cv_scores_rf.mean():.4f} +/- {cv_scores_rf.std():.4f}")
print("Random Forest trained")

---
## 🟠 Model 3: XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators=300, learning_rate=0.05,
    max_depth=6, subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbosity=0
)
xgb.fit(X_train_sel, y_train,
        eval_set=[(X_test_sel, y_test)], verbose=False)
cv_scores_xgb = cross_val_score(xgb, X_train_sel, y_train, cv=cv, scoring='r2')

y_pred_xgb_log  = xgb.predict(X_test_sel)
y_pred_xgb_euro = np.expm1(y_pred_xgb_log)

print(f"XGB CV R2: {cv_scores_xgb.mean():.4f} +/- {cv_scores_xgb.std():.4f}")
print("XGBoost trained")

---
## 📊 3. CV Comparison

In [ ]:
models_names = ['Ridge', 'Random Forest', 'XGBoost']
cv_means = [cv_scores_ridge.mean(), cv_scores_rf.mean(), cv_scores_xgb.mean()]
cv_stds  = [cv_scores_ridge.std(),  cv_scores_rf.std(),  cv_scores_xgb.std()]
colors   = ['#2E5D9E', '#27AE60', '#C55A11']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(models_names, cv_means, color=colors, edgecolor='white',
    alpha=0.9, yerr=cv_stds, capsize=6)
axes[0].set_title('5-Fold CV Mean R2 Score', fontweight='bold')
axes[0].set_ylabel('R2 Score')
axes[0].set_ylim(0, 1)
for bar, val in zip(bars, cv_means):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+0.02,
        f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

cv_data = [cv_scores_ridge, cv_scores_rf, cv_scores_xgb]
bp = axes[1].boxplot(cv_data, labels=models_names, patch_artist=True,
    medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_title('CV Score Distribution', fontweight='bold')
axes[1].set_ylabel('R2 Score')

plt.suptitle('Cross-Validation Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mt_01_cv_comparison.png', bbox_inches='tight')
plt.show()

---
## 🎯 4. Actual vs Predicted

In [ ]:
preds_euro = [y_pred_ridge_euro, y_pred_rf_euro, y_pred_xgb_euro]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name, pred, color in zip(axes, models_names, preds_euro, colors):
    ax.scatter(y_test_actual, pred, alpha=0.4, s=18, color=color)
    mn = min(y_test_actual.min(), pred.min())
    mx = max(y_test_actual.max(), pred.max())
    ax.plot([mn,mx],[mn,mx],'r--',linewidth=1.5,label='Perfect')
    r2 = r2_score(y_test_actual, pred)
    ax.set_title(f'{name} — Actual vs Predicted', fontweight='bold')
    ax.set_xlabel('Actual Price (EUR)')
    ax.set_ylabel('Predicted Price (EUR)')
    ax.text(0.05,0.92,f'R2={r2:.3f}',transform=ax.transAxes,fontsize=10,fontweight='bold',color=color)
    ax.legend(fontsize=8)

plt.suptitle('Actual vs Predicted (EUR) — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mt_02_actual_vs_pred.png', bbox_inches='tight')
plt.show()

---
## 📉 5. Residual Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name, pred, color in zip(axes, models_names, preds_euro, colors):
    residuals = y_test_actual.values - pred
    ax.scatter(pred, residuals, alpha=0.4, s=18, color=color)
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_title(f'{name} Residuals', fontweight='bold')
    ax.set_xlabel('Predicted Price (EUR)')
    ax.set_ylabel('Residual')

plt.suptitle('Residual Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mt_03_residuals.png', bbox_inches='tight')
plt.show()

---
## 🌟 6. Feature Importance

In [ ]:
ohe_features = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_cols).tolist()
all_feature_names = numerical_cols + ohe_features + binary_cols
selected_mask = selector.get_support()
selected_names = [f for f,m in zip(all_feature_names, selected_mask) if m]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, model, name, color in zip(axes, [rf, xgb], ['Random Forest','XGBoost'], ['#27AE60','#C55A11']):
    fi_df = pd.DataFrame({'Feature': selected_names, 'Importance': model.feature_importances_})
    fi_df = fi_df.sort_values('Importance', ascending=True).tail(15)
    ax.barh(fi_df['Feature'], fi_df['Importance'], color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{name} — Top 15 Feature Importances', fontweight='bold')
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mt_04_feature_importance.png', bbox_inches='tight')
plt.show()

---
## 📈 7. Train vs Test R2 (Overfitting Check)

In [ ]:
def get_r2(model, Xtr, ytr, Xte, yte):
    return r2_score(ytr, model.predict(Xtr)), r2_score(yte, model.predict(Xte))

r_ridge = get_r2(ridge, X_train_sel, y_train, X_test_sel, y_test)
r_rf    = get_r2(rf,    X_train_sel, y_train, X_test_sel, y_test)
r_xgb   = get_r2(xgb,  X_train_sel, y_train, X_test_sel, y_test)

x = np.arange(3)
width = 0.35
train_r2s = [r_ridge[0], r_rf[0], r_xgb[0]]
test_r2s  = [r_ridge[1], r_rf[1], r_xgb[1]]

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x-width/2, train_r2s, width, label='Train R2', color='#2E5D9E', alpha=0.85, edgecolor='white')
b2 = ax.bar(x+width/2, test_r2s,  width, label='Test R2',  color='#C55A11', alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(models_names)
ax.set_ylabel('R2 Score'); ax.set_ylim(0, 1.05)
ax.set_title('Train vs Test R2 — Overfitting Check', fontweight='bold')
ax.legend(); ax.bar_label(b1, fmt='%.3f', padding=3); ax.bar_label(b2, fmt='%.3f', padding=3)
plt.tight_layout()
plt.savefig('mt_05_train_vs_test.png', bbox_inches='tight')
plt.show()

print(f"{'Model':<20} {'Train R2':>10} {'Test R2':>10} {'Gap':>8}")
print("-"*50)
for name,(tr,te) in zip(models_names,[r_ridge,r_rf,r_xgb]):
    print(f"{name:<20} {tr:>10.4f} {te:>10.4f} {tr-te:>8.4f}")

---
## 💾 8. Save All Models

In [ ]:
joblib.dump(ridge,        'models/ridge_model.pkl')
joblib.dump(rf,           'models/rf_model.pkl')
joblib.dump(xgb,          'models/xgb_model.pkl')
joblib.dump(preprocessor, 'models/preprocessor.pkl')
joblib.dump(selector,     'models/feature_selector.pkl')
joblib.dump(xgb,          'models/model.pkl')   # best model

pd.Series(selected_names).to_csv('selected_features.csv', index=False, header=False)
pd.Series(numerical_cols).to_csv('numerical_cols.csv', index=False, header=False)
pd.Series(categorical_cols).to_csv('categorical_cols.csv', index=False, header=False)
pd.Series(binary_cols).to_csv('binary_cols.csv', index=False, header=False)

print("Saved: model.pkl, preprocessor.pkl, feature_selector.pkl")
print("Saved: ridge_model.pkl, rf_model.pkl, xgb_model.pkl")
print("Saved: col lists CSVs")

---
## ✅ 9. Summary & Download

In [ ]:
print("="*60)
print("         MODEL TRAINING SUMMARY")
print("="*60)
summary = [
    ("Ridge",          cv_scores_ridge.mean(), r_ridge[0], r_ridge[1]),
    ("Random Forest",  cv_scores_rf.mean(),    r_rf[0],    r_rf[1]),
    ("XGBoost",        cv_scores_xgb.mean(),   r_xgb[0],   r_xgb[1]),
]
print(f"  {'Model':<20} {'CV R2':>8} {'Train R2':>10} {'Test R2':>10}")
print(f"  {'-'*50}")
for name,cv,tr,te in summary:
    print(f"  {name:<20} {cv:>8.4f} {tr:>10.4f} {te:>10.4f}")

best = max(summary, key=lambda x: x[3])
print(f"\n Best Model: {best[0]}  (Test R2 = {best[3]:.4f})")
print("  Saved as: models/model.pkl")
print("\n Next Step: Model Evaluation (Step 7)")

from google.colab import files
for fname in ['models/model.pkl','models/preprocessor.pkl',
              'models/feature_selector.pkl','selected_features.csv',
              'numerical_cols.csv','categorical_cols.csv','binary_cols.csv']:
    files.download(fname)
print("All model files downloaded!")